# 🏦 Investment Portfolio Vault — AI Insights
## Gold Price Prediction Model (PortfolioInsightModel v1.1 — Bug Fixed)

**Pipeline Overview:**
1. Data Loading & Preprocessing
2. Feature Engineering (Lag, MA, Volatility, Returns, Volume)
3. Model Training (LSTM via TensorFlow/Keras)
4. Evaluation
5. 30-Day Price Prediction
6. Risk Score Calculation
7. Recommendation Engine
8. JSON Output Generation (strict schema)

---
**Dataset:** `Gold_data.csv` — Daily OHLCV from 2000-08-30 to 2024-06-24 (~5975 rows)

**Fixes applied (v1.1):**
- 🔴 Fix 1 — Scaler now fit only on training data (no data leakage)
- 🔴 Fix 2 — Forecast dates use business days only (pd.bdate_range)
- 🔴 Fix 3 — Forecast row update now refreshes lags, open/high/low
- 🟡 Fix 4 — predicted_price now included in JSON output
- 🟡 Fix 5 — Diversification label corrected
- 🟡 Fix 6 — MAPE formula unified via shared helper
- 🟢 Fix 7 — Volume added as a feature (vol_change)

## 📦 Step 0 — Install & Import Dependencies

In [ ]:
# Install any missing packages (Colab usually has all of these)
!pip install tensorflow scikit-learn pandas numpy matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import uuid
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print('All imports successful ✅')

## 📂 Step 1 — Load & Inspect Data

In [ ]:
# ── Upload Gold_data.csv in Colab, then run this cell ──
# from google.colab import files
# uploaded = files.upload()   # uncomment if uploading manually

# OR mount Google Drive:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Gold_data.csv'

#DATA_PATH = 'Gold_data.csv'   # adjust path if needed

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Shape      : {df.shape}')
print(f'Date range : {df["date"].iloc[0].date()} → {df["date"].iloc[-1].date()}')
print(f'Columns    : {list(df.columns)}')
df.head()

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())

## 🔧 Step 2 — Preprocessing & Feature Engineering

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build all ML features on top of raw OHLCV data.

    Features added:
      - daily_return      : percentage change in close price
      - log_return        : log of (close_t / close_{t-1})
      - ma_7 / ma_14 / ma_30 : simple moving averages
      - vol_7 / vol_14 / vol_30 : rolling std-dev (volatility proxy)
      - lag_1 … lag_5     : lagged close prices
      - price_range       : high - low (intraday range)
      - vol_change        : volume percentage change  [FIX 7 — added volume feature]
    """
    d = df.copy()

    # ── Returns ──────────────────────────────────────────────────
    d['daily_return'] = d['close'].pct_change()            # % change
    d['log_return']   = np.log(d['close'] / d['close'].shift(1))

    # ── Moving Averages ───────────────────────────────────────────
    for w in [7, 14, 30]:
        d[f'ma_{w}'] = d['close'].rolling(window=w).mean()

    # ── Rolling Volatility (std dev of returns) ───────────────────
    for w in [7, 14, 30]:
        d[f'vol_{w}'] = d['daily_return'].rolling(window=w).std()

    # ── Lag Features ─────────────────────────────────────────────
    for lag in range(1, 6):
        d[f'lag_{lag}'] = d['close'].shift(lag)

    # ── Intraday Range ────────────────────────────────────────────
    d['price_range'] = d['high'] - d['low']

    # ── Volume Feature [FIX 7] ────────────────────────────────────
    if 'volume' in d.columns:
        d['vol_change'] = d['volume'].pct_change()
    else:
        d['vol_change'] = 0.0   # fallback if volume column absent

    # Drop rows with NaNs introduced by rolling/shifting
    d = d.dropna().reset_index(drop=True)
    return d


df_feat = engineer_features(df)
print(f'After feature engineering: {df_feat.shape}')
df_feat.tail(3)

In [ ]:
# ── Plot historical close price ───────────────────────────────────
plt.figure(figsize=(14, 4))
plt.plot(df_feat['date'], df_feat['close'], color='goldenrod', linewidth=1)
plt.title('Gold Close Price — Historical', fontsize=14)
plt.xlabel('Date'); plt.ylabel('Price (USD)')
plt.tight_layout(); plt.show()

## 🧠 Step 3 — Prepare LSTM Dataset

In [ ]:
# ── Configuration ────────────────────────────────────────────────
SEQUENCE_LEN   = 60      # look-back window (trading days)
PREDICT_DAYS   = 30      # forecast horizon
TEST_SPLIT     = 0.15    # 15 % of data held out for evaluation
VALID_SPLIT    = 0.10    # 10 % of training for validation

# Feature columns fed into the LSTM
# [FIX 7] — vol_change (volume momentum) added
FEATURE_COLS = [
    'close', 'open', 'high', 'low',
    'daily_return', 'log_return',
    'ma_7', 'ma_14', 'ma_30',
    'vol_7', 'vol_14', 'vol_30',
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5',
    'price_range', 'vol_change'
]
TARGET_COL = 'close'

print(f'Feature count  : {len(FEATURE_COLS)}')
print(f'Sequence length: {SEQUENCE_LEN} days')
print(f'Predict horizon: {PREDICT_DAYS} days')

In [ ]:
# ── Scale features to [0, 1] ─────────────────────────────────────
# [FIX 1] — Scalers are now fit ONLY on training data to prevent data leakage.
#            Test data is transformed using the already-fitted scaler.

feature_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler  = MinMaxScaler(feature_range=(0, 1))

# Determine train/test boundary on the raw feature array
n_total     = len(df_feat)
train_size  = int(n_total * (1 - TEST_SPLIT))

raw_features = df_feat[FEATURE_COLS].values
raw_target   = df_feat[[TARGET_COL]].values

# Fit ONLY on train portion, then transform each split separately
scaled_features = np.zeros_like(raw_features, dtype=float)
scaled_features[:train_size] = feature_scaler.fit_transform(raw_features[:train_size])
scaled_features[train_size:] = feature_scaler.transform(raw_features[train_size:])

scaled_target = np.zeros((n_total, 1), dtype=float)
scaled_target[:train_size] = target_scaler.fit_transform(raw_target[:train_size])
scaled_target[train_size:] = target_scaler.transform(raw_target[train_size:])


def create_sequences(features, target, seq_len):
    """
    Convert flat arrays into (X, y) sequences for LSTM.
    X shape: (samples, seq_len, n_features)
    y shape: (samples,)
    """
    X, y = [], []
    for i in range(seq_len, len(features)):
        X.append(features[i - seq_len : i])
        y.append(target[i, 0])
    return np.array(X), np.array(y)


X_all, y_all = create_sequences(scaled_features, scaled_target, SEQUENCE_LEN)

# ── Train / Test split (time-aware — NO shuffling) ───────────────
split_idx = int(len(X_all) * (1 - TEST_SPLIT))
X_train_full, X_test = X_all[:split_idx], X_all[split_idx:]
y_train_full, y_test = y_all[:split_idx], y_all[split_idx:]

# ── Train / Validation split ─────────────────────────────────────
val_idx  = int(len(X_train_full) * (1 - VALID_SPLIT))
X_train, X_val = X_train_full[:val_idx], X_train_full[val_idx:]
y_train, y_val = y_train_full[:val_idx], y_train_full[val_idx:]

print(f'X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'X_val   : {X_val.shape}    |  y_val   : {y_val.shape}')
print(f'X_test  : {X_test.shape}   |  y_test  : {y_test.shape}')

## 🏗️ Step 4 — Build & Train LSTM Model

In [ ]:
def build_lstm_model(seq_len: int, n_features: int) -> tf.keras.Model:
    """
    Two-layer stacked LSTM with dropout for regularisation.
    Architecture:
        Input  → LSTM(128, return_sequences=True) → Dropout(0.2)
               → LSTM(64)                          → Dropout(0.2)
               → Dense(32, relu)                   → Dense(1)
    """
    model = Sequential([
        Input(shape=(seq_len, n_features)),

        # First LSTM layer — returns full sequence for the next LSTM
        LSTM(128, return_sequences=True),
        Dropout(0.2),

        # Second LSTM layer — returns only the last hidden state
        LSTM(64, return_sequences=False),
        Dropout(0.2),

        # Fully connected layers
        Dense(32, activation='relu'),
        Dense(1)   # single output: scaled close price
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='huber',          # Huber loss is robust to outliers vs MSE
        metrics=['mae']
    )
    return model


model = build_lstm_model(SEQUENCE_LEN, len(FEATURE_COLS))
model.summary()

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────
callbacks = [
    # Stop training if val_loss doesn't improve for 10 epochs
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    # Reduce LR by 50% if val_loss plateaus for 5 epochs
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ── Training curves ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss (Huber)'); axes[0].legend()

axes[1].plot(history.history['mae'],     label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('MAE'); axes[1].legend()

plt.tight_layout(); plt.show()

## 📊 Step 5 — Evaluate on Test Set

In [ ]:
# [FIX 6] — Shared MAPE helper used by both evaluate_model() and
#            compute_confidence_score() to ensure formula consistency.
def compute_mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Mean Absolute Percentage Error (safe division, returns %)."""
    return float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-9))) * 100)


def evaluate_model(model, X_test, y_test, target_scaler):
    """
    Returns actual vs predicted prices (in original USD scale)
    and prints standard regression metrics.
    """
    y_pred_scaled = model.predict(X_test, verbose=0).flatten()

    # Inverse-transform back to USD
    y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = compute_mape(y_true, y_pred)   # uses shared helper
    r2   = r2_score(y_true, y_pred)

    print('=' * 40)
    print('  MODEL EVALUATION (Test Set)')
    print('=' * 40)
    print(f'  MAE  : ${mae:,.2f}')
    print(f'  RMSE : ${rmse:,.2f}')
    print(f'  MAPE : {mape:.2f}%')
    print(f'  R²   : {r2:.4f}')
    print('=' * 40)

    return y_true, y_pred


y_true, y_pred = evaluate_model(model, X_test, y_test, target_scaler)

In [ ]:
# ── Actual vs Predicted plot ──────────────────────────────────────
plt.figure(figsize=(14, 4))
plt.plot(y_true, label='Actual',    color='goldenrod', linewidth=1.5)
plt.plot(y_pred, label='Predicted', color='steelblue', linewidth=1.5, linestyle='--')
plt.title('Gold Price — Actual vs Predicted (Test Set)')
plt.xlabel('Test Sample Index'); plt.ylabel('Price (USD)')
plt.legend(); plt.tight_layout(); plt.show()

## 🔮 Step 6 — 30-Day Future Price Prediction

In [ ]:
def predict_future_prices(
    model,
    df_feat: pd.DataFrame,
    feature_scaler,
    target_scaler,
    feature_cols: list,
    seq_len: int,
    n_days: int
) -> list:
    """
    Iteratively predicts `n_days` future closing prices using the
    last `seq_len` rows as the initial seed window.

    [FIX 3] — Synthetic rows now update open, high, low, and all
               lag features at each step, not just close.
    """
    df_run    = df_feat[feature_cols].copy()
    all_scaled = feature_scaler.transform(df_run.values)
    future_prices = []
    window = all_scaled[-seq_len:].copy()

    # Pre-compute column indices
    idx = {col: feature_cols.index(col) for col in feature_cols}

    for _ in range(n_days):
        X_input     = window[np.newaxis, :, :]
        pred_scaled = model.predict(X_input, verbose=0)[0, 0]
        pred_price  = target_scaler.inverse_transform([[pred_scaled]])[0, 0]
        future_prices.append(float(pred_price))

        # Build synthetic next row from last row
        new_row = window[-1].copy()

        # [FIX 3a] — Update price columns to predicted close
        new_row[idx['close']] = pred_scaled
        new_row[idx['open']]  = pred_scaled
        new_row[idx['high']]  = pred_scaled
        new_row[idx['low']]   = pred_scaled

        # [FIX 3b] — Shift lag features forward
        for lag in range(5, 1, -1):
            if f'lag_{lag}' in idx and f'lag_{lag-1}' in idx:
                new_row[idx[f'lag_{lag}']] = new_row[idx[f'lag_{lag-1}']]
        new_row[idx['lag_1']] = pred_scaled

        # Slide window forward
        window = np.vstack([window[1:], new_row])

    return future_prices


future_prices = predict_future_prices(
    model, df_feat, feature_scaler, target_scaler,
    FEATURE_COLS, SEQUENCE_LEN, PREDICT_DAYS
)

current_price       = float(df_feat['close'].iloc[-1])
predicted_price_30d = future_prices[-1]

print(f'Current price (last close)  : ${current_price:,.2f}')
print(f'Predicted price (day +30)   : ${predicted_price_30d:,.2f}')
print(f'Expected change             : {((predicted_price_30d - current_price) / current_price * 100):+.2f}%')

In [ ]:
# ── Plot forecast ─────────────────────────────────────────────────
last_date = df_feat['date'].iloc[-1]

# [FIX 2] — Use business days only (no weekends), matching gold trading calendar
future_dates = pd.bdate_range(
    start=last_date + timedelta(days=1), periods=PREDICT_DAYS
).tolist()

# Show last 120 days of history + 30-day forecast
hist_dates  = df_feat['date'].iloc[-120:].tolist()
hist_prices = df_feat['close'].iloc[-120:].tolist()

plt.figure(figsize=(14, 5))
plt.plot(hist_dates,   hist_prices,   label='Historical',       color='goldenrod', linewidth=1.5)
plt.plot(future_dates, future_prices, label='30-Day Forecast',  color='steelblue',
         linewidth=2, linestyle='--', marker='o', markersize=3)
plt.axvline(last_date, color='gray', linestyle=':', linewidth=1)
plt.title('Gold Price — 30-Day Forecast (Business Days)', fontsize=14)
plt.xlabel('Date'); plt.ylabel('Price (USD)')
plt.legend(); plt.tight_layout(); plt.show()

## ⚠️ Step 7 — Risk Score & Recommendation Engine

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#   RISK SCORE MODULE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def compute_risk_score(df_feat: pd.DataFrame, lookback: int = 90) -> dict:
    """
    Computes a volatility-based risk score (0–1) from the last
    `lookback` trading days.

    Method:
      1. Annualised volatility  = 30-day rolling std of daily returns × sqrt(252)
      2. Normalise to [0, 1]   by comparing against historical min/max
      3. Map to categorical     Low / Medium / High

    Thresholds (annualised vol):
      Low    < 10%
      Medium  10% – 20%
      High   > 20%
    """
    recent = df_feat.tail(lookback).copy()

    recent_vol_30 = recent['vol_30'].iloc[-1]
    annual_vol    = recent_vol_30 * np.sqrt(252)

    hist_vol          = df_feat['vol_30'].dropna()
    vol_min, vol_max  = hist_vol.min(), hist_vol.max()

    volatility_score = float(np.clip(
        (recent_vol_30 - vol_min) / (vol_max - vol_min + 1e-9), 0, 1
    ))
    volatility_score = round(volatility_score, 4)

    if annual_vol < 0.10:
        risk_level = 'Low'
    elif annual_vol < 0.20:
        risk_level = 'Medium'
    else:
        risk_level = 'High'

    return {
        'risk_level'       : risk_level,
        'volatility_score' : volatility_score,
        'annual_vol_pct'   : round(annual_vol * 100, 2)
    }


risk_info = compute_risk_score(df_feat)
print('Risk Assessment:')
for k, v in risk_info.items():
    print(f'  {k}: {v}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#   CONFIDENCE SCORE MODULE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def compute_confidence_score(y_true: np.ndarray, y_pred: np.ndarray,
                              volatility_score: float) -> float:
    """
    Confidence score = blend of model accuracy and market stability.

    Formula:
      accuracy_component  = 1 - MAPE/100          (clamped 0–1)
      stability_component = 1 - volatility_score  (lower vol → higher confidence)
      confidence          = 0.6 * accuracy + 0.4 * stability

    [FIX 6] — Uses shared compute_mape() for formula consistency.
    """
    mape                = compute_mape(y_true, y_pred)   # shared helper
    accuracy_component  = float(np.clip(1 - mape / 100, 0, 1))
    stability_component = float(1 - volatility_score)
    confidence          = 0.6 * accuracy_component + 0.4 * stability_component
    return round(float(np.clip(confidence, 0, 1)), 4)


confidence_score = compute_confidence_score(y_true, y_pred, risk_info['volatility_score'])
print(f'Confidence Score: {confidence_score:.4f}  ({confidence_score*100:.1f}%)')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#   TREND & RECOMMENDATION ENGINE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def compute_trend(expected_change_pct: float, threshold: float = 1.0) -> str:
    if expected_change_pct > threshold:
        return 'Uptrend'
    elif expected_change_pct < -threshold:
        return 'Downtrend'
    else:
        return 'Stable'


def generate_recommendation(expected_change_pct: float, risk_level: str) -> str:
    """
    Rule-based recommendation engine.
      1. Negative return                     → Sell
      2. High return (>3%) + Low/Medium risk → Buy
      3. All other positive cases            → Hold
    """
    if expected_change_pct < 0:
        return 'Sell'
    elif expected_change_pct >= 3.0 and risk_level in ('Low', 'Medium'):
        return 'Buy'
    else:
        return 'Hold'


expected_change_pct = round((predicted_price_30d - current_price) / current_price * 100, 4)
trend               = compute_trend(expected_change_pct)
recommendation      = generate_recommendation(expected_change_pct, risk_info['risk_level'])

print(f'Expected change  : {expected_change_pct:+.2f}%')
print(f'Trend            : {trend}')
print(f'Risk Level       : {risk_info["risk_level"]}')
print(f'Recommendation   : {recommendation}')

## 📤 Step 8 — Generate JSON Output (Strict Schema)

In [ ]:
def build_portfolio_insight_json(
    current_price        : float,
    predicted_price      : float,
    expected_change_pct  : float,
    trend                : str,
    confidence_score     : float,
    risk_level           : str,
    volatility_score     : float,
    recommendation       : str,
    user_id              : str = None
) -> dict:
    """
    Assembles the final output dict matching the PortfolioInsightModel
    JSON schema (version 1.1).

    [FIX 4] — predicted_price is now written into the JSON output.
    """
    now      = datetime.utcnow().isoformat() + 'Z'
    asset_id = str(uuid.uuid4())
    uid      = user_id or str(uuid.uuid4())

    overall_risk_map = {'Low': 'Low', 'Medium': 'Moderate', 'High': 'High'}
    overall_risk     = overall_risk_map.get(risk_level, 'Moderate')

    output = {
        'schema_version' : '1.1',
        'generated_at'   : now,
        'user': {
            'user_id': uid
        },
        'model_metadata': {
            'model_name'         : 'PortfolioInsightModel',
            'prediction_horizon' : '30_days',
            'model_version'      : 'v1.1'
        },
        'investment_insights': [
            {
                'asset_id'   : asset_id,
                'asset_name' : 'Gold',
                'asset_type' : 'Commodity',
                'market_data': {
                    'current_price': round(current_price, 2),
                    'last_updated' : now
                },
                'prediction': {
                    'predicted_price'        : round(predicted_price, 2),  # [FIX 4]
                    'trend'                  : trend,
                    'expected_change_percent': round(expected_change_pct, 4),
                    'confidence_score'       : round(confidence_score, 4)
                },
                'risk_metrics': {
                    'risk_level'      : risk_level,
                    'volatility_score': round(volatility_score, 4)
                },
                'recommendation': recommendation
            }
        ],
        'portfolio_insights': {
            'portfolio_value' : round(current_price, 2),
            'total_assets'    : 1,
            'performance': {
                'expected_growth_percent' : round(expected_change_pct, 4),
                'top_performing_asset'    : 'Gold'
            },
            'risk_analysis': {
                'overall_risk_level' : overall_risk,
                'highest_risk_asset' : 'Gold'
            },
            'diversification': {
                'diversification_score': 1
            }
        }
    }
    return output


output_json = build_portfolio_insight_json(
    current_price        = current_price,
    predicted_price      = predicted_price_30d,
    expected_change_pct  = expected_change_pct,
    trend                = trend,
    confidence_score     = confidence_score,
    risk_level           = risk_info['risk_level'],
    volatility_score     = risk_info['volatility_score'],
    recommendation       = recommendation,
    user_id              = 'user_demo_001'
)

print(json.dumps(output_json, indent=2))

In [ ]:
# ── Save JSON to file ─────────────────────────────────────────────
OUTPUT_PATH = 'portfolio_insight_output.json'
with open(OUTPUT_PATH, 'w') as f:
    json.dump(output_json, f, indent=2)

print(f'✅  Output saved to: {OUTPUT_PATH}')

# Download in Colab
# from google.colab import files
# files.download(OUTPUT_PATH)

## 🔍 Step 9 — Summary Dashboard

In [ ]:
insight = output_json['investment_insights'][0]
port    = output_json['portfolio_insights']

print('=' * 55)
print('  📊  PORTFOLIO INSIGHT SUMMARY')
print('=' * 55)
print(f"  Asset            : {insight['asset_name']} ({insight['asset_type']})")
print(f"  Current Price    : ${insight['market_data']['current_price']:,.2f}")
print(f"  Predicted (30d)  : ${insight['prediction']['predicted_price']:,.2f}")
print(f"  Expected Change  : {insight['prediction']['expected_change_percent']:+.2f}%")
print(f"  Trend            : {insight['prediction']['trend']}")
print(f"  Confidence       : {insight['prediction']['confidence_score'] * 100:.1f}%")
print('-' * 55)
print(f"  Risk Level       : {insight['risk_metrics']['risk_level']}")
print(f"  Volatility Score : {insight['risk_metrics']['volatility_score']:.4f}  (0=low, 1=high)")
print(f"  Annual Vol       : {risk_info['annual_vol_pct']:.2f}%")
print('-' * 55)
print(f"  ⚡  RECOMMENDATION  :  {insight['recommendation']}")
print('=' * 55)
print(f"  Portfolio Value  : ${port['portfolio_value']:,.2f}")
print(f"  Overall Risk     : {port['risk_analysis']['overall_risk_level']}")
# [FIX 5] — Clarified label: it's the number of assets, not a rating out of 10
print(f"  Diversification  : {port['diversification']['diversification_score']} asset(s) (single-asset portfolio)")
print('=' * 55)

## 🚀 Step 10 — Scaling to Multiple Assets (Design Guide)

To extend this pipeline from Gold → N assets:

```
ASSET REGISTRY
┌──────────────────────────────────────────────────────┐
│  assets = [                                          │
│    {'id': 'gold_001',   'name': 'Gold',              │
│     'type': 'Commodity','csv': 'Gold_data.csv'},     │
│    {'id': 'silver_001', 'name': 'Silver',            │
│     'type': 'Commodity','csv': 'Silver_data.csv'},   │
│    {'id': 'tcs_001',    'name': 'TCS',               │
│     'type': 'Stock',   'csv': 'TCS_data.csv'},       │
│  ]                                                   │
└──────────────────────────────────────────────────────┘
```

**Pipeline changes needed:**

1. **Separate model per asset** — train one LSTM per asset CSV, save with `model.save(f'{asset_id}_model.h5')`
2. **Loop over assets** — wrap Steps 2–7 in `for asset in assets:` and collect results into a list
3. **Portfolio-level aggregation:**
   - `portfolio_value` = sum of all asset values
   - `overall_risk` = weighted average of individual risk scores
   - `diversification_score` = number of assets (or Herfindahl index for weights)

---
**End of Notebook** — `PortfolioInsightModel v1.1`  
Gold · LSTM · 30-Day Horizon · Risk + Recommendation Engine  
*All 7 bugs from v1.0 have been fixed.*